In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# NOTEBOOK 05: GOLD TRANSCRIPT EVALUATION
#
# First time in this project we measure ASR quality against what pilots
# ACTUALLY SAID rather than what they SHOULD have said (ADS-B proxy).
#
# Gold transcripts: 6 WAV files manually transcribed
#   3.wav, 6.wav, 7.wav, 8.wav, 10.wav, 12.wav
#
# Three comparisons:
#   A0: Raw Whisper-small vs gold      (true baseline WER)
#   A1: Mistral-corrected vs gold      (true GEC improvement)
#   Proxy: ADS-B expected vs gold      (how wrong were proxy metrics?)
# ═══════════════════════════════════════════════════════════════════════════════

import ast
import re
import json
import warnings
from pathlib import Path
from difflib import SequenceMatcher

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from jiwer import wer, cer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
plt.style.use("seaborn-v0_8-whitegrid")

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT     = Path("C:/xcas-ga-comms-assistant")
INTERIM  = ROOT / "data/interim"
OUTPUTS  = ROOT / "outputs"

MANIFEST_CSV  = INTERIM / "manifest_2020-10-22.csv"
SEGMENTS_JSONL= OUTPUTS / "segments_small_atco2.jsonl"
A1_CSV        = OUTPUTS / "corrected_approach1_small_atco2 (1).csv"
ADSB_CSV      = INTERIM / "adsb_with_callouts_2020-10-22.csv"

# ── Gold files ────────────────────────────────────────────────────────────────
GOLD_FILES = ["3.wav","6.wav","7.wav","8.wav","10.wav","12.wav"]

# Verify all files exist
for p in [MANIFEST_CSV, SEGMENTS_JSONL, A1_CSV, ADSB_CSV]:
    status = "✅" if p.exists() else "❌ MISSING"
    print(f"  {status}  {p.name}")

print("\n✅ All paths configured")

  ✅  manifest_2020-10-22.csv
  ✅  segments_small_atco2.jsonl
  ✅  corrected_approach1_small_atco2 (1).csv
  ✅  adsb_with_callouts_2020-10-22.csv

✅ All paths configured


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# LOAD ALL DATA
# ═══════════════════════════════════════════════════════════════════════════════

# ── Manifest with gold transcripts ────────────────────────────────────────────
df_manifest = pd.read_csv(MANIFEST_CSV)

# Parse aircraft_in_window: stored as "['EJM410', 'N410LG', 'N6886D']"
def parse_aircraft_list(val):
    try:
        return ast.literal_eval(str(val)) if pd.notna(val) else []
    except:
        return [v.strip().strip("'") for v in
                str(val).strip("[]").split(",") if v.strip()]

df_manifest["aircraft_list"] = df_manifest["aircraft_in_window"].apply(
    parse_aircraft_list
)

# Focus on gold files only
df_gold_files = df_manifest[
    df_manifest["wav_file"].isin(GOLD_FILES)
].set_index("wav_file")

print("MANIFEST — Gold files loaded:")
for wav in GOLD_FILES:
    if wav in df_gold_files.index:
        row  = df_gold_files.loc[wav]
        gt   = str(row.get("ground_truth",""))
        segs = [s.strip() for s in gt.split("|") if s.strip()]
        print(f"  {wav:8s}: {len(segs):2d} gold segments  "
              f"| aircraft: {row['aircraft_list']}")

# ── JSONL segments ────────────────────────────────────────────────────────────
print("\nLoading JSONL segments...")
jsonl_records = {}
with open(SEGMENTS_JSONL) as f:
    for line in f:
        rec = json.loads(line)
        wav = rec["wav_file"]
        if wav in GOLD_FILES:
            # Only clean, ATC-keyword segments
            clean = [
                s for s in rec["segments"]
                if not s.get("is_hallucination", True)
                and s.get("has_atc_keyword", False)
                and s.get("text","").strip()
            ]
            jsonl_records[wav] = clean
            print(f"  {wav:8s}: {len(clean):2d} clean JSONL segments")

# ── A1 corrected output ────────────────────────────────────────────────────────
print("\nLoading A1 corrected outputs...")
df_a1 = pd.read_csv(A1_CSV)
df_a1_gold = df_a1[df_a1["wav_file"].isin(GOLD_FILES)].copy()
print(f"  A1 rows for gold files: {len(df_a1_gold)}")

# ── ADS-B expected callouts ────────────────────────────────────────────────────
print("\nLoading ADS-B callouts (for proxy comparison)...")
df_adsb = pd.read_csv(ADSB_CSV, parse_dates=["timestamp"])
print(f"  ADS-B rows: {len(df_adsb):,}")

# ── Semantic embedding model ───────────────────────────────────────────────────
print("\nLoading semantic model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ all-MiniLM-L6-v2 ready")

MANIFEST — Gold files loaded:
  3.wav   :  1 gold segments  | aircraft: ['MLN285', 'N248CG', 'N285DX']
  6.wav   : 20 gold segments  | aircraft: ['LXJ421', 'N13337', 'N40WV', 'N421FX', 'N6683S', 'UAL2089']
  7.wav   : 13 gold segments  | aircraft: ['FDY277', 'LXJ421', 'N53226', 'N6683S', 'N932SP']
  8.wav   : 33 gold segments  | aircraft: ['AAL745', 'FDY333', 'N13337', 'N53226', 'N600AL', 'N6683S', 'N6886D']
  10.wav  :  1 gold segments  | aircraft: ['EJM410', 'N410LG', 'N6886D']
  12.wav  :  1 gold segments  | aircraft: ['N1824H', 'N56987', 'N6683S', 'N92141', 'N990JB']

Loading JSONL segments...
  10.wav  :  3 clean JSONL segments
  12.wav  : 13 clean JSONL segments
  3.wav   :  1 clean JSONL segments
  6.wav   : 14 clean JSONL segments
  7.wav   : 11 clean JSONL segments
  8.wav   : 33 clean JSONL segments

Loading A1 corrected outputs...
  A1 rows for gold files: 16

Loading ADS-B callouts (for proxy comparison)...
  ADS-B rows: 219,622

Loading semantic model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ all-MiniLM-L6-v2 ready


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# GOLD TRANSCRIPT PARSING
#
# Each gold transcript is pipe-separated in one CSV cell.
# Some segments are noise/PTT clicks — detect and label those.
# Normalise text for fair metric comparison.
# ═══════════════════════════════════════════════════════════════════════════════

# ATC keywords that confirm real speech
ATC_KW = [
    "butler","runway","traffic","inbound","outbound","downwind","base",
    "final","crosswind","upwind","departing","landing","takeoff","touch",
    "pattern","teardrop","miles","north","south","east","west","cessna",
    "piper","warrior","cherokee","november","alpha","bravo","charlie",
    "delta","echo","foxtrot","golf","hotel","india","juliet","kilo",
    "lima","mike","oscar","papa","quebec","romeo","sierra","tango",
    "uniform","victor","whiskey","xray","yankee","zulu","rolling",
    "staying","full","stop","departure","overflying","crosswind",
]

def is_noise_segment(text: str) -> bool:
    """
    Detect PTT clicks, static, and non-speech segments in gold transcripts.
    These are real entries but not transcribable speech.
    """
    t = text.lower().strip()
    if not t or len(t) < 3:
        return True
    # PTT clicks
    if re.search(r"(cl[i]?ck|click){2,}", t, re.IGNORECASE):
        return True
    # Only punctuation or special chars
    if re.match(r"^[^a-z]+$", t):
        return True
    # Very short with no ATC content
    words = t.split()
    if len(words) <= 2 and not any(kw in t for kw in ATC_KW):
        return True
    return False


def normalise(text: str) -> str:
    """Lowercase, strip punctuation, collapse whitespace."""
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def parse_gold_file(wav_file: str) -> list[dict]:
    """
    Parse gold transcript for one WAV file.
    Returns list of dicts with: idx, raw, normalised, is_noise,
    runway, callsign_fragment.
    """
    row = df_gold_files.loc[wav_file]
    raw_gt = str(row.get("ground_truth",""))
    segments_raw = [s.strip() for s in raw_gt.split("|") if s.strip()]

    rwy_pat = re.compile(
        r"\brunway\s+(two six|zero eight|26|08|2 6|0 8)\b",
        re.IGNORECASE
    )

    results = []
    for i, seg in enumerate(segments_raw):
        norm = normalise(seg)
        rwy_matches = rwy_pat.findall(seg)
        results.append({
            "gold_idx"     : i,
            "raw"          : seg,
            "normalised"   : norm,
            "is_noise"     : is_noise_segment(seg),
            "runway_in_gold": rwy_matches[0].lower() if rwy_matches else None,
            "word_count"   : len(norm.split()),
        })
    return results


# Parse all gold files and report
all_gold = {}
print("GOLD TRANSCRIPT ANALYSIS")
print("="*55)
for wav in GOLD_FILES:
    segs = parse_gold_file(wav)
    speech = [s for s in segs if not s["is_noise"]]
    noise  = [s for s in segs if s["is_noise"]]
    rwy_segs = [s for s in speech if s["runway_in_gold"]]
    all_gold[wav] = segs

    print(f"\n{wav}:")
    print(f"  Total segments  : {len(segs)}")
    print(f"  Speech segments : {len(speech)}")
    print(f"  Noise/click segs: {len(noise)}"
          + (f"  ← {[s['raw'][:30] for s in noise]}" if noise else ""))
    print(f"  With runway ref : {len(rwy_segs)}")
    if speech:
        sample = speech[0]["raw"]
        print(f"  Sample (seg 1)  : '{sample[:70]}'")

GOLD TRANSCRIPT ANALYSIS

3.wav:
  Total segments  : 1
  Speech segments : 1
  Noise/click segs: 0
  With runway ref : 1
  Sample (seg 1)  : 'Take off on runway two six southbound departure butler county'

6.wav:
  Total segments  : 20
  Speech segments : 20
  Noise/click segs: 0
  With runway ref : 3
  Sample (seg 1)  : 'Butler Traffic Cessna Eight Three Sierra departing runway two six stay'

7.wav:
  Total segments  : 13
  Speech segments : 13
  Noise/click segs: 0
  With runway ref : 5
  Sample (seg 1)  : 'Butler Traffic Cessna Eight Three sierra left base two six butler'

8.wav:
  Total segments  : 33
  Speech segments : 33
  Noise/click segs: 0
  With runway ref : 19
  Sample (seg 1)  : 'Butler Traffic Cessna Eight Three sierra clear of the active butler'

10.wav:
  Total segments  : 1
  Speech segments : 1
  Noise/click segs: 0
  With runway ref : 1
  Sample (seg 1)  : 'butler traffic cessna two two six is base runway 26 butler. Butler tra'

12.wav:
  Total segments  : 1
  Speech

In [6]:
# ── ALIGNMENT DIAGNOSTIC ──────────────────────────────────────────────────────
# Shows similarity scores for first file to understand why matches are failing

diag_wav = "8.wav"   # use your largest file
gold_segs  = [s for s in all_gold[diag_wav] if not s["is_noise"]]
jsonl_segs = jsonl_records.get(diag_wav, [])

print(f"ALIGNMENT DIAGNOSTIC — {diag_wav}")
print(f"Gold speech segments : {len(gold_segs)}")
print(f"JSONL clean segments : {len(jsonl_segs)}")
print(f"\nFirst 5 JSONL vs best gold match:")
print("-"*70)

for j_idx, jseg in enumerate(jsonl_segs[:5]):
    scores = []
    for g_idx, gseg in enumerate(gold_segs):
        s = similarity_score(jseg["text"], gseg["raw"])
        scores.append((s, g_idx, gseg["raw"]))
    scores.sort(reverse=True)
    best_s, best_idx, best_gold = scores[0]
    second_s = scores[1][0] if len(scores) > 1 else 0

    print(f"\nJSONL[{j_idx}] (start={jseg.get('start',0):.1f}s):")
    print(f"  ASR : '{jseg['text'][:65]}'")
    print(f"  BEST gold[{best_idx}] sim={best_s:.3f}: '{best_gold[:65]}'")
    print(f"  2nd best sim={second_s:.3f}")
    print(f"  Would align at threshold 0.12: {'YES' if best_s>=0.12 else 'NO'}")

ALIGNMENT DIAGNOSTIC — 8.wav
Gold speech segments : 33
JSONL clean segments : 33

First 5 JSONL vs best gold match:
----------------------------------------------------------------------

JSONL[0] (start=25.0s):
  ASR : 'Gateway traffic Cessna Eight Three Sierra cleared to be active ri'
  BEST gold[10] sim=0.670: 'Butler Traffic Cessna Eight Three sierra left crosswind two six b'
  2nd best sim=0.667
  Would align at threshold 0.12: YES

JSONL[1] (start=79.1s):
  ASR : 'Evett traffic suspect that we do shift to downwind runway two six'
  BEST gold[22] sim=0.712: 'butler traffic cessna two two six is downwind runway two six butl'
  2nd best sim=0.712
  Would align at threshold 0.12: YES

JSONL[2] (start=142.9s):
  ASR : 'Roger traffic Red Hat Three Two Six is based runway two six roger'
  BEST gold[13] sim=0.746: 'butler traffic cessna two two six is base runway two six butler'
  2nd best sim=0.746
  Would align at threshold 0.12: YES

JSONL[3] (start=170.0s):
  ASR : 'Oscar traffic Swi

In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — REVISED ALIGNMENT USING HUNGARIAN ALGORITHM
#
# Problem with sliding window: one bad early match (JSONL[0]→gold[10])
# shifts all subsequent search windows, causing cascade failures.
#
# 📚 Hungarian Algorithm:
#   Given n JSONL segments and m gold segments, build an n×m similarity
#   matrix. Find the assignment of JSONL→gold that maximises TOTAL
#   similarity across all pairs simultaneously.
#   
#   For 8.wav (33 vs 33): instead of greedily assigning JSONL[0]→gold[10]
#   and leaving gold[0-9] orphaned, the global solver considers all 33×33=1089
#   combinations and finds the overall optimal matching.
#
# scipy.optimize.linear_sum_assignment implements this in O(n³) time.
# For our sizes (max 37 segments) this is instantaneous.
# ═══════════════════════════════════════════════════════════════════════════════

from scipy.optimize import linear_sum_assignment

def similarity_score(asr_text: str, gold_text: str) -> float:
    a = normalise(asr_text)
    g = normalise(gold_text)
    if not a or not g: return 0.0

    # Char-level
    char_sim = SequenceMatcher(None, a, g).ratio()

    # ATC keyword overlap
    a_kw = set(kw for kw in ATC_KW if kw in a)
    g_kw = set(kw for kw in ATC_KW if kw in g)
    kw_sim = (len(a_kw & g_kw) / len(a_kw | g_kw)
              if (a_kw | g_kw) else 0.0)

    # Runway agreement
    rwy_pat = re.compile(r"(two six|zero eight|runway)", re.IGNORECASE)
    a_rwy = set(rwy_pat.findall(a))
    g_rwy = set(rwy_pat.findall(g))
    rwy_match = 1.0 if (a_rwy and g_rwy and a_rwy == g_rwy) else 0.0

    # Word-level overlap
    stopwords = {"a","the","to","of","for","is","in","at","and","or","it","s"}
    a_words = set(a.split()) - stopwords
    g_words = set(g.split()) - stopwords
    word_overlap = (len(a_words & g_words) / len(a_words | g_words)
                    if (a_words | g_words) else 0.0)

    return (0.25*char_sim + 0.30*kw_sim +
            0.20*rwy_match + 0.25*word_overlap)


def align_segments(wav_file: str,
                   min_similarity: float = 0.10) -> list[dict]:
    """
    Global optimal alignment using Hungarian algorithm.

    Steps:
    1. Build full similarity matrix (n_jsonl × n_gold)
    2. Solve optimal assignment (maximise total similarity)
    3. Remove assignments below min_similarity threshold
    4. Verify monotonic ordering — reject pairs that violate time order
    """
    gold_segs  = [s for s in all_gold[wav_file] if not s["is_noise"]]
    jsonl_segs = jsonl_records.get(wav_file, [])
    if not jsonl_segs or not gold_segs:
        return []

    n_j = len(jsonl_segs)
    n_g = len(gold_segs)

    # ── Step 1: Full similarity matrix ────────────────────────────────────────
    sim_matrix = np.zeros((n_j, n_g))
    for i, jseg in enumerate(jsonl_segs):
        for j, gseg in enumerate(gold_segs):
            sim_matrix[i, j] = similarity_score(
                jseg["text"], gseg["raw"]
            )

    # ── Step 2: Hungarian assignment (minimise cost = 1 - similarity) ─────────
    # Pad to square matrix — Hungarian requires square input
    n_max = max(n_j, n_g)
    cost  = np.ones((n_max, n_max))   # padding cost = 1.0 (worst)
    cost[:n_j, :n_g] = 1.0 - sim_matrix

    row_ind, col_ind = linear_sum_assignment(cost)

    # ── Step 3: Extract valid assignments ─────────────────────────────────────
    # Only real rows/cols (not padding) with similarity above threshold
    assignment = {}   # jsonl_idx → gold_idx
    for r, c in zip(row_ind, col_ind):
        if r < n_j and c < n_g:
            sim = sim_matrix[r, c]
            if sim >= min_similarity:
                assignment[r] = c

    # ── Step 4: Enforce monotonic ordering ────────────────────────────────────
    # Assignments must be non-decreasing in gold index
    # (JSONL seg 5 cannot map to gold seg 3 if JSONL seg 3 maps to gold seg 8)
    # Sort by JSONL index, keep only pairs that don't violate order
    sorted_assignments = sorted(assignment.items())  # [(jsonl_idx, gold_idx)]
    monotonic = {}
    last_gold  = -1
    for j_idx, g_idx in sorted_assignments:
        if g_idx > last_gold:
            monotonic[j_idx] = g_idx
            last_gold = g_idx
        # If order violated, discard this assignment
        # (the gold segment was already "used" earlier in time)

    # ── Build result list ──────────────────────────────────────────────────────
    pairs = []
    for j_idx, jseg in enumerate(jsonl_segs):
        g_idx   = monotonic.get(j_idx)
        aligned = g_idx is not None

        if aligned:
            g   = gold_segs[g_idx]
            sim = sim_matrix[j_idx, g_idx]
        else:
            g   = {"gold_idx":-1, "raw":"", "normalised":"",
                   "runway_in_gold":None}
            sim = float(np.max(sim_matrix[j_idx]))  # best possible (rejected)

        pairs.append({
            "wav_file"        : wav_file,
            "jsonl_idx"       : j_idx,
            "jsonl_start_s"   : jseg.get("start", 0),
            "jsonl_end_s"     : jseg.get("end",   0),
            "asr_text"        : jseg["text"],
            "gold_idx"        : g_idx if aligned else -1,
            "gold_raw"        : g.get("raw",""),
            "gold_normalised" : g.get("normalised",""),
            "runway_in_gold"  : g.get("runway_in_gold"),
            "similarity"      : round(sim, 3),
            "aligned"         : aligned,
            "reject_reason"   : ("" if aligned else
                                 f"best_sim={sim:.3f} or order_violated"),
        })

    return pairs


# ── Run revised alignment ──────────────────────────────────────────────────────
all_pairs = []
print("ALIGNMENT RESULTS — Hungarian Algorithm")
print("="*60)

for wav in GOLD_FILES:
    pairs   = align_segments(wav, min_similarity=0.08)
    all_pairs.extend(pairs)

    n_aligned   = sum(1 for p in pairs if p["aligned"])
    n_unaligned = len(pairs) - n_aligned
    avg_sim     = np.mean([p["similarity"] for p in pairs
                           if p["aligned"]] or [0])

    print(f"\n  {wav:8s}:  {n_aligned}/{len(pairs)} aligned  "
          f"avg_sim={avg_sim:.3f}")

    # Show first 2 aligned pairs as sanity check
    for p in [x for x in pairs if x["aligned"]][:2]:
        print(f"    ASR : '{p['asr_text'][:60]}'")
        print(f"    GOLD: '{p['gold_raw'][:60]}'")
        print(f"    Sim : {p['similarity']:.3f}")
        print()

    # Show unaligned and why
    if n_unaligned > 0:
        unaligned = [p for p in pairs if not p["aligned"]]
        print(f"    Unaligned ({n_unaligned}):")
        for p in unaligned[:2]:
            print(f"      ASR: '{p['asr_text'][:55]}'  "
                  f"[best_sim={p['similarity']:.3f}]")

df_pairs      = pd.DataFrame(all_pairs)
aligned_pairs = df_pairs[df_pairs["aligned"]].copy()

total_pct = 100*len(aligned_pairs)/max(len(df_pairs),1)
print(f"\n{'='*60}")
print(f"Total aligned : {len(aligned_pairs)} / {len(df_pairs)} "
      f"({total_pct:.0f}%)")
print(f"Target        : ≥55 pairs (73%+)")

if total_pct < 60:
    print(f"\n⚠  Still below target — check similarity scores:")
    print(f"   Distribution of best achievable similarity:")
    best_per_seg = [
        float(np.max(
            [[similarity_score(
                jsonl_records[r["wav_file"]][r["jsonl_idx"]]["text"],
                g["raw"]
              ) for g in [s for s in all_gold[r["wav_file"]]
                          if not s["is_noise"]]]
             for _ in [None]][0]
        ))
        for _, r in df_pairs.iterrows()
        if r["wav_file"] in jsonl_records
        and r["jsonl_idx"] < len(jsonl_records.get(r["wav_file"],[]))
    ]
    if best_per_seg:
        print(f"   Mean best-possible sim : {np.mean(best_per_seg):.3f}")
        print(f"   Min best-possible sim  : {np.min(best_per_seg):.3f}")
        print(f"   % above 0.10           : "
              f"{100*np.mean([s>0.10 for s in best_per_seg]):.0f}%")

ALIGNMENT RESULTS — Hungarian Algorithm

  3.wav   :  1/1 aligned  avg_sim=0.420
    ASR : 'Golf Bravo runway two six five point five roger cocks'
    GOLD: 'Take off on runway two six southbound departure butler count'
    Sim : 0.420


  6.wav   :  8/14 aligned  avg_sim=0.779
    ASR : 'Bolta traffic Cessna Eight Three Sierra departing runway two'
    GOLD: 'Butler Traffic Cessna Eight Three Sierra departing runway tw'
    Sim : 0.954

    ASR : 'Butler Regional Staff One Three Three Three Seven is departi'
    GOLD: 'Butler regional skyhawk One Three three three seven is depar'
    Sim : 0.734

    Unaligned (6):
      ASR: 'Bravo traffic Cessna Eight Three Sierra final two six C'  [best_sim=0.807]
      ASR: 'answer the first number just to ground-fault work count'  [best_sim=0.416]

  7.wav   :  2/11 aligned  avg_sim=0.698
    ASR : 'Butler traffic SNA Eight Three CRL left please two six Butle'
    GOLD: 'Butler Traffic Cessna Eight Three sierra left base two six b'
    Sim : 0.67

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# METRIC COMPUTATION AGAINST GOLD TRANSCRIPTS
# ═══════════════════════════════════════════════════════════════════════════════

def normalise_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compute_wer(ref: str, hyp: str) -> float:
    r, h = normalise_text(ref), normalise_text(hyp)
    if not r: return 0.0 if not h else 1.0
    return wer(r, h)

def compute_cer(ref: str, hyp: str) -> float:
    r, h = normalise_text(ref), normalise_text(hyp)
    if not r: return 0.0 if not h else 1.0
    return cer(r, h)

def compute_semwer(ref: str, hyp: str) -> float:
    if not ref.strip() or not hyp.strip():
        return 1.0
    embs = embed_model.encode(
        [normalise_text(ref), normalise_text(hyp)],
        convert_to_numpy=True, show_progress_bar=False
    )
    return float(1 - cosine_similarity(embs[0:1], embs[1:2])[0][0])

ENTITY_PATTERNS = {
    "runway"  : re.compile(r"\brunway\s+(two six|zero eight|26|08)\b", re.I),
    "position": re.compile(
        r"\b(downwind|base|final|crosswind|upwind|inbound|departing"
        r"|overflying|teardrop|pattern|touch and go|full stop|rolling"
        r"|staying in the pattern)\b", re.I),
    "airport" : re.compile(r"\bbutler\b", re.I),
}

def extract_entities(text: str) -> dict:
    t = normalise_text(text)
    return {k: list(set(m.lower() for m in p.findall(t)))
            for k, p in ENTITY_PATTERNS.items()}

def entity_error_rate(ref_ent: dict, hyp_ent: dict) -> float:
    WEIGHTS = {"runway":3.0, "position":1.0, "airport":0.5}
    tw, te = 0.0, 0.0
    for k, w in WEIGHTS.items():
        rv = set(ref_ent.get(k,[]))
        hv = set(hyp_ent.get(k,[]))
        if not rv: continue
        err = 0.0 if rv==hv else (1.0 if not hv else
              1.0 - len(rv&hv)/len(rv|hv))
        tw += w; te += w * err
    return te/tw if tw > 0 else 0.0

def aviation_semwer(ref: str, hyp: str) -> dict:
    esem = compute_semwer(ref, hyp)
    re_  = extract_entities(ref)
    he_  = extract_entities(hyp)
    eer  = entity_error_rate(re_, he_)
    return {
        "aviation_semwer" : 0.4*esem + 0.6*eer,
        "embed_semwer"    : esem,
        "entity_err_rate" : eer,
        "ref_runway"      : re_.get("runway",[]),
        "hyp_runway"      : he_.get("runway",[]),
    }


# ── Get A1 corrected text for each aligned pair ────────────────────────────────
# Match A1 results to aligned pairs by wav_file + nearest seg_start_s
def get_a1_text(wav_file: str, seg_start_s: float) -> str:
    subset = df_a1_gold[df_a1_gold["wav_file"] == wav_file].copy()
    if len(subset) == 0:
        return ""
    subset["dist"] = abs(subset["seg_start_s"] - seg_start_s)
    closest = subset.loc[subset["dist"].idxmin()]
    if closest["dist"] > 5.0:   # more than 5s away → no match
        return ""
    return str(closest.get("final_text","")) if pd.notna(
        closest.get("final_text")
    ) else ""


# ── Compute all metrics for aligned pairs ─────────────────────────────────────
print("Computing metrics for all aligned pairs...")
metric_rows = []

for _, pair in aligned_pairs.iterrows():
    ref  = pair["gold_normalised"]   # gold = true reference
    asr  = pair["asr_text"]          # raw Whisper output
    a1   = get_a1_text(pair["wav_file"], pair["jsonl_start_s"])
    rwy_gold = pair["runway_in_gold"]

    if not ref or not asr:
        continue

    m_asr = aviation_semwer(ref, asr)
    m_a1  = aviation_semwer(ref, a1) if a1 else None

    # Runway accuracy
    rwy_asr = m_asr["hyp_runway"]
    rwy_a1  = aviation_semwer(ref, a1)["hyp_runway"] if a1 else []

    def rwy_correct(rwy_list, gold_rwy):
    # Handle None and float NaN from pandas
        if gold_rwy is None:
            return None
        if isinstance(gold_rwy, float):
            return None   # NaN from pandas
        if not str(gold_rwy).strip():
            return None
        gold_norm = normalise_text(str(gold_rwy))
        return (any(gold_norm in normalise_text(r) or
                    normalise_text(r) in gold_norm
                    for r in rwy_list)
                if rwy_list else False)

    metric_rows.append({
        "wav_file"          : pair["wav_file"],
        "gold_segment"      : pair["gold_raw"][:80],
        "asr_text"          : asr[:80],
        "a1_text"           : a1[:80] if a1 else "",
        "alignment_sim"     : pair["similarity"],
        # ASR vs gold
        "wer_asr"           : compute_wer(ref, asr),
        "cer_asr"           : compute_cer(ref, asr),
        "avsem_asr"         : m_asr["aviation_semwer"],
        "embed_sem_asr"     : m_asr["embed_semwer"],
        "eer_asr"           : m_asr["entity_err_rate"],
        # A1 vs gold
        "wer_a1"            : compute_wer(ref, a1)    if a1 else float("nan"),
        "cer_a1"            : compute_cer(ref, a1)    if a1 else float("nan"),
        "avsem_a1"          : m_a1["aviation_semwer"] if m_a1 else float("nan"),
        "embed_sem_a1"      : m_a1["embed_semwer"]    if m_a1 else float("nan"),
        "eer_a1"            : m_a1["entity_err_rate"] if m_a1 else float("nan"),
        # Deltas
        "avsem_delta"       : (m_a1["aviation_semwer"] -
                               m_asr["aviation_semwer"]) if m_a1 else float("nan"),
        "wer_delta"         : (compute_wer(ref,a1) -
                               compute_wer(ref,asr)) if a1 else float("nan"),
        # Runway accuracy
        "runway_in_gold"    : rwy_gold,
        "runway_asr_correct": rwy_correct(rwy_asr, rwy_gold),
        "runway_a1_correct" : rwy_correct(rwy_a1,  rwy_gold),
    })

df_metrics = pd.DataFrame(metric_rows)
df_metrics.to_csv(OUTPUTS/"gold_evaluation_metrics.csv", index=False)
print(f"✅ Metrics computed for {len(df_metrics)} aligned pairs")
print(f"   Saved → gold_evaluation_metrics.csv")

Computing metrics for all aligned pairs...


AttributeError: 'float' object has no attribute 'lower'